# Heft 2: Ein Signal neben seiner Erwartung

Ein Signal ist eine Größe, die das System erwartet: mit Bedeutung, Einheit, plausiblem Bereich und Betriebsbereich.
Dieses Heft legt die Daten eines Signals daneben: die Rohreihe und die bereinigte Reihe übereinander, die Werte
außerhalb des plausiblen Bereichs markiert, den Betriebsbereich als Band. Am Ende steht, was die Entscheidung der
Wasserstoffkette heute über ihre Eingänge sagt.

Das Heft liest nur. Es braucht den Retrieval-Server (Weg A).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from opentsx.client import RetrievalClient

RETRIEVAL = os.environ.get("OPENTSX_RETRIEVAL", "http://opentsx-retrieval.opentsx.svc:8080")
SYSTEM = "h2-chain"
client = RetrievalClient(RETRIEVAL)
print("Retrieval-Server:", RETRIEVAL)

## 1. Die Signale des Systems

Eine Zeile je Signal: Rolle, Zustand, Herkunft, Erwartung und die Reihen, die das System für die Güten `raw` und
`clean` gefunden hat.

In [ ]:
signals = client.signals(SYSTEM)
signals[["id", "role", "state", "origin", "unit", "min", "max", "operating_min", "operating_max", "interval", "next"]]

## 2. Ein Signal wählen

Vorgewählt ist der Wind der DWD-Station am Park: gemessen in 10 m Höhe, roh und bereinigt vorhanden. Andere
Signale mit Daten: `park.p_potential_kw` (geschätzt aus dem Wind), `market.price_eur_mwh` (gemeldet),
`offtake.demand_kgph` (erzeugt, ein Demo-Generator).

In [ ]:
SIGNAL = "park/DWD-01759/wind_speed"
START, END = "2025-10-01T00:00:00Z", "2025-11-01T00:00:00Z"

sig = signals.set_index("id").loc[SIGNAL]
print(f"{SIGNAL}: {sig['meaning']}")
print(f"Erwartung: {sig['unit']}, plausibel {sig['min']} bis {sig['max']}, Betrieb {sig['operating_min']} bis {sig['operating_max']}, Takt {sig['interval']}")
print(f"Zustand {sig['state']}, Herkunft {sig['origin']}")


def series_of(ids):
    if not isinstance(ids, str) or not ids:
        return pd.Series(dtype=float)
    f = client.frame(series=ids.split(","), start=START, end=END)
    return f.set_index("ts")["value"].sort_index() if len(f) else pd.Series(dtype=float)


raw = series_of(sig["raw_series"])
clean = series_of(sig["clean_series"])
print(f"roh: {len(raw)} Werte, bereinigt: {len(clean)} Werte zwischen {START} und {END}")

## 3. Roh und bereinigt übereinander

Grau die Rohreihe, blau die bereinigte. Rote Punkte sind Rohwerte außerhalb des plausiblen Bereichs; die
Bereinigung macht aus ihnen Lücken und rechnet sie nicht schön. Das Band ist der Betriebsbereich: Werte außerhalb
davon sind möglich, aber kein Normalbetrieb.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
if pd.notna(sig["operating_min"]) and pd.notna(sig["operating_max"]):
    ax.axhspan(sig["operating_min"], sig["operating_max"], color="#e3efe3", label="Betriebsbereich")
if len(raw):
    raw.plot(ax=ax, lw=0.5, color="#8a949e", label="roh")
if len(clean):
    clean.plot(ax=ax, lw=0.8, color="#1f6fb3", label="bereinigt")
if len(raw) and pd.notna(sig["min"]) and pd.notna(sig["max"]):
    out = raw[(raw < sig["min"]) | (raw > sig["max"])]
    ax.scatter(out.index, out.values, color="#b3392b", s=12, zorder=3, label=f"außerhalb plausibel ({len(out)})")
ax.set_ylabel(sig["unit"] or "")
ax.set_title(SIGNAL)
ax.legend(loc="upper right")
plt.show()

## 4. Was die Bereinigung geändert hat

Gleiche Zeitpunkte nebeneinander: wie viele Werte gleich geblieben sind, wie viele zur Lücke wurden, und die
größten Unterschiede.

In [ ]:
if len(raw) and len(clean):
    both = pd.concat({"roh": raw, "bereinigt": clean}, axis=1)
    same = (both["roh"] - both["bereinigt"]).abs() < 1e-9
    print(f"gleich: {int(same.sum())}, zur Lücke geworden: {int((both['roh'].notna() & both['bereinigt'].isna()).sum())}, "
          f"nur bereinigt (anderer Takt): {int((both['roh'].isna() & both['bereinigt'].notna()).sum())}")
    display(both[~same].dropna(how="all").head(10))
else:
    print("Für dieses Signal gibt es nicht beide Güten im gewählten Zeitraum.")

## 5. Was die Entscheidung über ihre Eingänge sagt

Die Entscheidungskarte der Wasserstoffkette nennt die Signale, die sie braucht, und was sie für diesen Zweck tauglich
macht: Güte, Alter, zugelassene Herkunft. Daraus folgt für jede Analyse `usable`, `restricted` oder `blocked`, mit
Grund.

In [ ]:
status = client.decisions(SYSTEM)
for d in status["decisions"]:
    print(f"{d['title']} (Stand der Daten {d.get('asOf', 'n/a')})")
    print(pd.DataFrame([{"signal": n["signal"], "güte": n.get("grade", "clean"), "tauglich": n["ok"],
                         "grund": "; ".join(n["reasons"])} for n in d["needs"]]).to_string(index=False))
    for a in d["analyses"]:
        print(f"\n  {a['title']}: {a['status']}")
        for r in a["reasons"]:
            print(f"    - {r}")

Die Freigabe hier liest das Systemdokument ohne die Arbeitskopie eines Kontos. Was in der Web-Anwendung gebunden
wurde, sieht man auf dem Ort "Models".

Quellen: Deutscher Wetterdienst, CC BY 4.0; Bundesnetzagentur | SMARD.de, CC BY 4.0.